# GEAP Evaluation SDK demo — the eval + monitoring flywheel

Design metrics → score the deployed engine → publish two honest monitored surfaces → regression / simulated / offline-over-BQ → failure clustering → optimizer. SDK-first on *this repo's* modules.

**Honesty note (important).** The managed Agent Engine runtime strips prompt/response content from ADK traces, so the *native* Vertex Online Evaluators always return `INSUFFICIENT_DATA` (verified — see `docs/notes/offline-eval-monitoring-bridge.md`). This notebook therefore **drops the native online-monitor path** and uses the repo's **offline-eval bridge** as the canonical source. Rubric cells need a *deployed* engine (`AGENT_ENGINE_ID`).

## Setup

In [ ]:
# Repo-root bootstrap so `src.*` imports resolve from notebooks/demo/
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

def enabled(flag: str) -> bool:
    # Costly/live cells are opt-in via GEAP_RUN_* env flags (default off).
    return os.environ.get(flag, '0') in ('1', 'true', 'True')

import vertexai
from src import config
vertexai.init(project=config.GCP_PROJECT_ID, location=config.GCP_REGION)
AGENT_RESOURCE = (
    f'projects/{config.GCP_PROJECT_ID}/locations/{config.GCP_REGION}'
    f'/reasoningEngines/{config.AGENT_ENGINE_ID}'
)
print('project:', config.GCP_PROJECT_ID, '| region:', config.GCP_REGION)
print('coordinator engine:', config.AGENT_ENGINE_ID)

## Phase 1 — Design metrics (canonical names)

The two monitored surfaces are kept on separate axes so a task executor and an economic optimizer are never scored the same way. Canonical metric names live in `src/eval/quality_alerts.py` so nothing drifts.

In [ ]:
from src.eval.quality_alerts import ALL_MONITORED_METRICS, ROUTER_MONITORED_METRICS
print('coordinator quality (agent_eval/*, 1-5, alert <3.0):')
for name, thr in ALL_MONITORED_METRICS:
    print(f'  {name:<28} alert < {thr}')
print('router efficiency (agent_router/*, native units):')
for name, thr, cmp in ROUTER_MONITORED_METRICS:
    print(f'  {name:<28} alert {cmp} {thr}')

## Phase 2a — Rapid eval against the deployed engine (guarded)

`run_multi_agent_batch_eval` scores the *deployed* coordinator via the Vertex Gen AI Evaluation Service (6 rubrics). `--limit` / `limit=` caps cases for a fast pass. Set `GEAP_RUN_EVAL=1` (billable; needs a reachable engine).

In [ ]:
from src.eval.multi_agent_batch_eval import run_multi_agent_batch_eval
batch_results = None
if enabled('GEAP_RUN_EVAL'):
    batch_results = run_multi_agent_batch_eval(
        agents=['coordinator_agent'], agent_id=config.AGENT_ENGINE_ID, limit=8,
    )
    print(list(batch_results.get('agents', {})))
else:
    print('skipped — set GEAP_RUN_EVAL=1. '
          'CLI: uv run python -m src.eval.multi_agent_batch_eval '
          '--agents coordinator_agent --agent-id <ENGINE_ID> --limit 8')

## Phase 2a-publish — scores → Cloud Monitoring (offline bridge)

Instead of a raw `monitoring_v3` writer, delegate to the repo's bridge: `publish_offline_scores` extracts the three monitored quality metrics, scales 0-1 → 1-5, tags `eval_mode=offline`, and writes to `custom.googleapis.com/agent_eval/*`. Read back with `verify_monitors`.

In [ ]:
from src.eval.publish_offline_eval import publish_offline_scores
if enabled('GEAP_RUN_EVAL') and batch_results:
    written = publish_offline_scores(batch_results)
    print('published to agent_eval/*:', written)
else:
    print('skipped publish — needs a batch_results dict from Phase 2a')

In [ ]:
# Read both monitored surfaces back (canonical source):
import subprocess
if enabled('GEAP_RUN_EVAL'):
    print(subprocess.run(
        [sys.executable, '-m', 'src.eval.verify_monitors', '--format', 'json'],
        cwd=str(ROOT), capture_output=True, text=True).stdout[:2000])
else:
    print('CLI: uv run python -m src.eval.verify_monitors --format json')

## Phase 2b — Regression suite

A curated corpus (multi-step + adversarial) drives every rubric run.

In [ ]:
from src.eval.batch_eval import EVAL_CASES
print('regression cases:', len(EVAL_CASES))
print('example:', {k: EVAL_CASES[0][k] for k in list(EVAL_CASES[0])[:2]})

## Phase 2c — Simulated (multi-turn)

`build_agent_info` constructs the `AgentInfo` descriptor used for offline scoring; `src.eval.simulated_eval` drives a multi-turn user-simulator run against a deployed engine.

In [ ]:
from src.eval.agent_eval_configs import build_agent_info
info = build_agent_info('coordinator_agent')
print('AgentInfo for:', getattr(info, 'name', 'coordinator_agent'))
print('CLI: uv run python -m src.eval.simulated_eval --agent-id <ENGINE_ID> '
      '--agent-name coordinator_agent')

## Phase 2e — Offline over BigQuery traces

Content logged to BigQuery (opt-in `BigQueryAgentAnalyticsPlugin`, dataset `BQ_EVAL_DATASET`) can be re-scored offline. This BQ path exists precisely *because* the managed runtime strips trace content — see the honesty note.

In [ ]:
print('BQ eval dataset:', config.BQ_EVAL_DATASET)
print('agent-analytics dataset:', config.BQ_AGENT_ANALYTICS_DATASET,
      '| enabled:', config.ENABLE_AGENT_ANALYTICS)

## Router efficiency — the distinct second surface

The 5-tier router is scored on **native units**, not a 1-5 rubric: `routing_accuracy_pct`, `cost_savings_pct` (vs all-Opus), and `classifier_latency_ms` → `custom.googleapis.com/agent_router/*`.

In [ ]:
from src.eval.complexity_metrics import run_complexity_accuracy_eval, run_cost_efficiency_eval
from src.eval.publish_router_efficiency import publish_router_efficiency
from src.eval.agent_eval_configs import get_eval_cases
if enabled('GEAP_RUN_EVAL'):
    cases = get_eval_cases('router_agent')
    acc = await run_complexity_accuracy_eval(cases)     # noqa: F704
    cost = await run_cost_efficiency_eval(cases)        # noqa: F704
    print('published to agent_router/*:', publish_router_efficiency(acc, cost))
else:
    print('skipped — set GEAP_RUN_EVAL=1. '
          'CLI: uv run python -m src.eval.publish_router_efficiency --from-json <full_results.json>')

## ~~Native Online Monitors~~ → dropped (platform-blocked)

The fork's native online-evaluator path (`aiplatform.googleapis.com/online_evaluator/scores`) always returns `INSUFFICIENT_DATA` in this runtime, because prompt/response content is stripped from Agent Engine traces. The honest replacement is the offline bridge above: two series (`coordinator_quality` + `router_efficiency`) read by `verify_monitors`. See `docs/notes/offline-eval-monitoring-bridge.md`.

## Quality-drift alerts (custom metric types)

Alerts fire on the custom series, not the dead online-evaluator metric. Policies are parametrized in `quality_alerts.py`; lifecycle via `src.eval.manage_monitors`.

In [ ]:
print('CLI: uv run python -m src.eval.quality_alerts all   # one policy per monitored metric')
print('CLI: uv run python -m src.eval.manage_monitors      # list / prune alert policies')

## Failure clustering

Group low-scoring cases to see *where* the agent fails.

In [ ]:
from src.eval.failure_clusters import analyze_failure_clusters
if enabled('GEAP_RUN_EVAL'):
    print(analyze_failure_clusters(config.AGENT_ENGINE_ID))
else:
    print('skipped — set GEAP_RUN_EVAL=1 (needs eval history for the engine)')

## GEPA optimizer (guarded)

The GEPA prompt optimizer improves an agent's instruction from eval feedback. Heavy; opt-in. `src.optimize.run_optimize` applies ADK compatibility patches before running.

In [ ]:
from src.agents.coordinator.agent import root_agent as coordinator_root
print('optimize target:', coordinator_root.name)
if enabled('GEAP_RUN_OPT'):
    import subprocess
    subprocess.run([sys.executable, '-m', 'src.optimize.run_optimize',
                    'src/agents/coordinator'], cwd=str(ROOT), check=False)
else:
    print('skipped — set GEAP_RUN_OPT=1. '
          'CLI: uv run python -m src.optimize.run_optimize src/agents/coordinator')

## Recap

L1-native `client.evals.*` designs and runs rubrics; this repo's custom code turns them into two honest monitored surfaces (offline bridge, because the native online path is platform-blocked), adds a router-efficiency series the fork lacks, and closes the loop with clustering + GEPA optimization.